# SeaDronesSee ConvNeXt PT2E QAT — Kaggle

Notebook này chạy luồng PT2E graph-mode cho ConvNeXt backbone, rồi convert và benchmark INT8 trên CPU.

Điểm chính:

- PT2E chỉ quantize ConvNeXt body mặc định (`quantization.pt2e.region = backbone`)
- train theo từng epoch nhỏ để dễ resume trên Kaggle
- cuối cùng benchmark FP32 vs PT2E INT8


In [ ]:
import os
import sys
import json
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

import yaml

REPO = Path('/kaggle/working/EchteAI')
REPO_URL = os.environ.get('ECHTEAI_REPO_URL', 'https://github.com/NguyenDucThang-tb/EchteAI.git')
if not REPO.exists():
    print(f'Repo not found at {REPO}; cloning from {REPO_URL} ...', flush=True)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    print(f'Repo already present at {REPO}', flush=True)

sys.path.insert(0, str(REPO))

print('Repo:', REPO)


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '-e', '.[coco,pt2e]'
], check=True, cwd=REPO)

import importlib.metadata as ilm
import torch

print('Dependencies installed')
print('torch:', ilm.version('torch'))
print('torchvision:', ilm.version('torchvision'))
print('torchao:', ilm.version('torchao'))
assert tuple(int(part) for part in torch.__version__.split('+', 1)[0].split('.')[:3]) >= (2, 11, 0), \
    f'PT2E notebook requires torch >= 2.11.0, found {torch.__version__}'
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))


In [ ]:
from pipelines.convnext_qat.checkpoint import checkpoint_size_mb


def run_and_log(command, log_path, cwd=REPO):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', ' '.join(command), flush=True)
    print('Persistent log:', log_path, flush=True)
    print('Started:', datetime.now().isoformat(timespec='seconds'), flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'
===== START {datetime.now().isoformat()} =====
')
        log_file.write(' '.join(command) + '
')
        log_file.flush()
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env, cwd=cwd,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
        log_file.write(f'===== END code={return_code} {datetime.now().isoformat()} =====
')
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def checkpoint_epoch(path):
    path = Path(path)
    if not path.is_file():
        return 0
    payload = torch.load(path, map_location='cpu', weights_only=False)
    return int(payload.get('epoch', 0)) if isinstance(payload, dict) else 0


def find_latest_checkpoint(root, filename):
    root = Path(root)
    matches = list(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Không tìm thấy {filename} trong {root}')
    scored = []
    for path in matches:
        try:
            scored.append((checkpoint_epoch(path), path))
        except Exception:
            continue
    if not scored:
        raise FileNotFoundError(f'Không đọc được checkpoint nào cho {filename} trong {root}')
    scored.sort(key=lambda item: (item[0], str(item[1])))
    return scored[-1][1]


def find_dataset_root(root):
    root = Path(root)
    matches = list(root.rglob('instances_train.json'))
    if not matches:
        raise FileNotFoundError(f'Không tìm thấy instances_train.json trong {root}')
    for ann in matches:
        candidate = ann.parent.parent
        if (candidate / 'images' / 'train').exists() and (candidate / 'images' / 'val').exists():
            return candidate
    return matches[0].parent.parent


In [ ]:
DATA_ROOT = find_dataset_root('/kaggle/input')
FP32_CKPT_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/nguyenducthangtb/echteai-seadronessee-m3-checkpoints'),
    Path('/kaggle/input/echteai-seadronessee-m3-checkpoints'),
    Path('/kaggle/input'),
]
for _ckpt_root in FP32_CKPT_ROOT_CANDIDATES:
    if _ckpt_root.exists():
        try:
            FP32_SOURCE = find_latest_checkpoint(_ckpt_root, 'fp32_last.pt')
            break
        except FileNotFoundError:
            continue
else:
    raise FileNotFoundError('Không tìm thấy checkpoint fp32_last.pt trong dataset Kaggle đã train 5 epoch')
SELECTIVE_INT8_SOURCE = None
try:
    SELECTIVE_INT8_SOURCE = find_latest_checkpoint('/kaggle/input', 'selective_int8.pt')
except FileNotFoundError:
    pass

print('DATA_ROOT:', DATA_ROOT)
print('Using FP32 source:', FP32_SOURCE)
print('FP32 epoch:', checkpoint_epoch(FP32_SOURCE))
print('Optional selective INT8 source:', SELECTIVE_INT8_SOURCE)
assert checkpoint_epoch(FP32_SOURCE) >= 5, 'Cần checkpoint FP32 epoch 5 hoặc lớn hơn'


In [ ]:
WORK = Path('/kaggle/working/seadronessee_pt2e_run')
OUTPUT = WORK / 'checkpoints'
LOGS = WORK / 'logs'
OUTPUT.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

PT2E_BATCH_PER_GPU = 1
PT2E_GLOBAL_BATCH = PT2E_BATCH_PER_GPU * max(torch.cuda.device_count(), 1)

base = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
base['dataset'].update({
    'train_images': str(DATA_ROOT / 'images/train'),
    'train_annotations': str(DATA_ROOT / 'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT / 'images/val'),
    'val_annotations': str(DATA_ROOT / 'annotations/instances_val.json'),
    'test_images': str(DATA_ROOT / 'images/val'),
    'test_annotations': str(DATA_ROOT / 'annotations/instances_val.json'),
})
base['dataset']['workers'] = 1
base['model']['min_size'] = 768
base['model']['train_min_sizes'] = [640, 704, 768]
base['model']['max_size'] = 1280
base['training']['batch_size'] = PT2E_BATCH_PER_GPU
base['training']['fp32_batch_size'] = PT2E_BATCH_PER_GPU
base['training']['qat_batch_size'] = PT2E_BATCH_PER_GPU
base['training']['fp32_epochs'] = 5
base['training']['pt2e_qat_epochs'] = 1
base['training']['epoch_benchmark_images'] = 100
base['quantization']['variant'] = 'M3'
base['quantization']['pt2e']['region'] = 'backbone'
base['quantization']['pt2e']['observer_warmup_epochs'] = 0
base['quantization']['pt2e']['observer_freeze_epochs'] = 1
base['quantization']['pt2e']['example_batch_size'] = PT2E_BATCH_PER_GPU
base['quantization']['pt2e']['maximum_batch_size'] = PT2E_BATCH_PER_GPU
base['output'] = {
    'directory': str(OUTPUT),
    'fp32_best': str(FP32_SOURCE),
    'fp32_last': str(FP32_SOURCE),
    'pt2e_qat_best': str(OUTPUT / 'pt2e_qat_best.pt'),
    'pt2e_qat_last': str(OUTPUT / 'pt2e_qat_last.pt'),
    'pt2e_int8_model': str(OUTPUT / 'pt2e_int8.pt'),
    'pt2e_int8_evaluation': str(OUTPUT / 'pt2e_int8_evaluation.json'),
    'pt2e_benchmark_json': str(OUTPUT / 'pt2e_benchmark.json'),
}
RUNTIME_CONFIG = WORK / 'runtime.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False), encoding='utf-8')

print('Runtime config:', RUNTIME_CONFIG)
print('PT2E batch_per_gpu:', PT2E_BATCH_PER_GPU)
print('PT2E global_batch:', PT2E_GLOBAL_BATCH)
print(RUNTIME_CONFIG.read_text())
print('PT2E output dir:', OUTPUT)


In [ ]:
def run_pt2e_one_epoch(epoch_idx):
    pt2e_last = OUTPUT / 'pt2e_qat_last.pt'
    command = [
        sys.executable, '-u', 'scripts/train_next_epoch.py',
        '--config', str(RUNTIME_CONFIG),
        '--stage', 'pt2e',
    ]
    if pt2e_last.exists():
        command += ['--resume', str(pt2e_last)]
    print(f'PT2E epoch {epoch_idx} command ready')
    run_and_log(command, LOGS / 'pt2e_train.log', cwd=REPO)

if torch.cuda.device_count() >= 2:
    print('PT2E training will run with DDP on 2 GPUs when available.')
else:
    print('PT2E training will run on a single GPU in this session.')


In [ ]:
# Chay dung 1 epoch PT2E roi dung lai de convert va benchmark 100 anh
print('===== PT2E epoch 1/1 =====')
run_pt2e_one_epoch(1)
pt2e_last = OUTPUT / 'pt2e_qat_last.pt'
print('Checkpoint epoch:', checkpoint_epoch(pt2e_last))
print('Checkpoint size MB:', f'{checkpoint_size_mb(pt2e_last):.2f}')


In [ ]:
# Convert PT2E QAT checkpoint tot nhat sang INT8 artifact va luu lai
from pipelines.convnext_qat.checkpoint import load_checkpoint, checkpoint_size_mb
from pipelines.convnext_qat.config import load_config
from pipelines.convnext_qat.models import build_fasterrcnn_convnext
from pipelines.convnext_qat.quantization import (
    convert_pt2e_backbone,
    prepare_pt2e_backbone_qat,
    save_pt2e_int8_artifact,
    set_pt2e_qat_phase,
)

pt2e_last = OUTPUT / 'pt2e_qat_last.pt'
pt2e_source = pt2e_last
pt2e_int8 = OUTPUT / 'pt2e_int8.pt'

assert pt2e_source.is_file(), f'Khong tim thay PT2E checkpoint de convert: {pt2e_source}'

config = load_config(str(RUNTIME_CONFIG), require_dataset=True)
model = build_fasterrcnn_convnext(config)
model = prepare_pt2e_backbone_qat(model, config)
payload = load_checkpoint(pt2e_source, model, map_location='cpu')
set_pt2e_qat_phase(model, 'frozen')
model.eval()

int8_model = convert_pt2e_backbone(model, inplace=False, compile_region=False)
save_pt2e_int8_artifact(
    pt2e_int8,
    int8_model,
    metrics=payload.get('metrics', {}),
    extra={
        'source_qat': str(pt2e_source),
        'source_epoch': payload.get('epoch', 0),
        'region': getattr(model, 'pt2e_quantized_region', 'backbone.body'),
    },
)

print('Converted from:', pt2e_source)
print('Saved INT8 artifact:', pt2e_int8)
print('INT8 artifact size MB:', f'{checkpoint_size_mb(pt2e_int8):.2f}')


In [ ]:
# Benchmark 100 anh: FP32 vs PT2E INT8 tren CPU
import gc
import json

from evaluate import load_model
from pipelines.convnext_qat.checkpoint import model_state_size_mb
from pipelines.convnext_qat.data import build_coco_loader
from pipelines.convnext_qat.metrics import evaluate_model
from pipelines.convnext_qat.quantization import load_pt2e_int8_artifact

MAX_IMAGES = 100
BENCHMARK_JSON = OUTPUT / 'pt2e_fp32_100_images.json'
pt2e_int8 = OUTPUT / 'pt2e_int8.pt'
assert pt2e_int8.is_file(), f'Khong tim thay PT2E INT8 artifact: {pt2e_int8}'

config = load_config(str(RUNTIME_CONFIG), require_dataset=True)
torch.set_num_threads(int(config.get('benchmark', {}).get('num_threads', 1)))

def benchmark_100_images(model, loader, device, max_images=100):
    was_training = model.training
    model.eval()
    processed = 0
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    started = time.perf_counter()
    for images, _ in loader:
        if processed >= max_images:
            break
        images = [img.to(device) for img in images[: max_images - processed]]
        model(images)
        processed += len(images)
        if processed % 10 == 0 or processed >= max_images:
            print(f'benchmark progress: {processed}/{max_images} images', flush=True)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    seconds = time.perf_counter() - started
    if was_training:
        model.train()
    return {
        'images': processed,
        'seconds': seconds,
        'latency_ms_per_image': 1000.0 * seconds / processed,
        'fps': processed / seconds,
        'device': str(device),
    }

results = {}

print('Loading FP32...', flush=True)
fp32 = load_model(config, 'fp32', str(FP32_SOURCE), torch.device('cpu'))
results['fp32'] = benchmark_100_images(
    fp32, build_coco_loader(config, 'test', shuffle=False, limit=MAX_IMAGES, batch_size=1), torch.device('cpu'), MAX_IMAGES
)
print('Evaluating FP32 mAP...', flush=True)
results['fp32']['accuracy'] = evaluate_model(
    fp32,
    build_coco_loader(config, 'test', shuffle=False, limit=MAX_IMAGES, batch_size=1),
    torch.device('cpu'),
    include_rpn=True,
)
results['fp32']['model_size_mb'] = model_state_size_mb(fp32)
del fp32
gc.collect()

print('Loading PT2E INT8...', flush=True)
pt2e, artifact = load_pt2e_int8_artifact(str(pt2e_int8), config)
results['pt2e_int8'] = benchmark_100_images(
    pt2e, build_coco_loader(config, 'test', shuffle=False, limit=MAX_IMAGES, batch_size=1), torch.device('cpu'), MAX_IMAGES
)
print('Evaluating PT2E INT8 mAP...', flush=True)
results['pt2e_int8']['accuracy'] = evaluate_model(
    pt2e,
    build_coco_loader(config, 'test', shuffle=False, limit=MAX_IMAGES, batch_size=1),
    torch.device('cpu'),
    include_rpn=False,
)
results['pt2e_int8']['model_size_mb'] = model_state_size_mb(pt2e)
results['pt2e_int8']['artifact_metadata'] = artifact.get('extra', {})
results['speedup_vs_fp32'] = results['fp32']['latency_ms_per_image'] / results['pt2e_int8']['latency_ms_per_image']

BENCHMARK_JSON.write_text(json.dumps(results, indent=2), encoding='utf-8')
print(json.dumps(results, indent=2))
print('Saved:', BENCHMARK_JSON)


## Ghi chú

- Notebook này tập trung vào PT2E graph-mode để tối ưu inference speed.
- Default region là `backbone`; nhánh `backbone_fpn` vẫn có thể bật trong `runtime.yaml` nhưng chưa phải default ổn định nhất.
- Workflow mặc định ở đây là: train 1 epoch -> convert từ `pt2e_qat_last.pt` -> benchmark `FP32 vs PT2E INT8` trên 100 ảnh.
- PT2E train đã có nhánh DDP; nếu session Kaggle có 2 GPU thì `train_next_epoch.py` sẽ tự động chuyển sang `train_pt2e_qat_ddp.py`.
